# Retail Store Sales Performance Dashboard

### Dataset
Online Retail II Dataset (UCI) — archive.ics.uci.edu/dataset/502/online+retail+ii — 1M+ transactions.

### Context
The data consists of all the transactions occurring for a UK-based and registered, non-store online retail between 01/12/2009 and 09/12/2011 stored in an excel file. Eight variables are presented for analysis: Invoice No, StockCode, Description, Quantity, Invoice Date, Unit Price, Customer ID, and Country.  

### Key Questions
- Which products generate the most revenue?
- Who are the top 10 customers?
- What does monthly revenue seasonality look like?

In [ ]:
import sys
print(sys.executable)
import os
import logging
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
print(openpyxl.__version__)

# ETL folder layers (Curated Layering)
RAW_DIR = '../data/raw'
STAGING_DIR = '../data/staging'
CURATED_DIR = '../data/curated'
LOG_DIR = '../logs'
os.makedirs(STAGING_DIR, exist_ok=True)
os.makedirs(CURATED_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Logging setup — writes to both the notebook output and a permanent log file
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(f'{LOG_DIR}/pipeline.log', mode='w'), # mode='w' overwrites the log each time its run
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


## Stage 1A (Extraction):  Load dataset

In [ ]:
# Load excel file and save back as a csv file for faster download in the future. 
transactions_raw = pd.read_excel('../data/raw/online_retail_II.xlsx')
transactions_raw.to_csv('../data/raw/online_retail_II.csv', index=False)

# Preview loaded data frame
print(transactions_raw.head())

## Stage 1B (Extraction):  Inspect dataset

In [ ]:
# Show shape and column list of loaded dataframe
print(transactions_raw.shape)
print()
print(transactions_raw.columns.tolist())

### Interpretation: transactions_raw.shape and .columns.tolist()
#### Shape
525,461 rows × 8 columns.
#### Columns
Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country

#### Observations
- 8 columns total: 3 likely identifiers (Invoice, StockCode, Customer ID).
- Column naming is inconsistent in style — most are single words in
  CamelCase (InvoiceDate, StockCode), while Customer ID uses a space.
  Worth keeping in mind for bracket-access syntax (transactions_raw['Customer ID']
  requires quotes/brackets rather than dot-notation, unlike the others).
- Row count (525,461) sets the scale for interpreting all later counts —
  e.g. any null count or duplicate count found later should be read as a
  fraction of this total.

In [ ]:
# Show dataset structure and datatypes
transactions_raw.info()

### Interpretation: transactions_raw.info()

#### Row / column count
525,461 entries (RangeIndex 0 to 525460), 8 columns — consistent with ``.shape``.

#### Non-null counts and dtypes, per column
| Column | Non-Null Count | Dtype |
|---|---|---|
| Invoice | 525,461 | object |
| StockCode | 525,461 | object |
| Description | 522,533 | object |
| Quantity | 525,461 | int64 |
| InvoiceDate | 525,461 | datetime64[us] |
| Price | 525,461 | float64 |
| Customer ID | 417,534 | float64 |
| Country | 525,461 | str |

#### Completeness
- Fully populated: ``Invoice, StockCode, Quantity, InvoiceDate, Price, Country``
- ``Description``: 522,533 non-null → 2,928 missing
- ``Customer ID``: 417,534 non-null → 107,927 missing — largest gap of any column

#### Dtype observations
- ``InvoiceDate`` is already datetime64[us], not object — no parse_dates
  conversion needed at load time.
- ``Customer ID`` is float64, not int64 or object — likely a side effect of
  the missing values (a numeric column with NaN present cannot be int64
  in pandas without an explicit nullable-int dtype).
- Memory usage: 32.1+ MB for this DataFrame.

#### Anomalies flagged for follow-up (not yet investigated)
- ``Invoice``, ``StockCode``, and ``Description`` all show dtype object, while ``Country``
  shows dtype ``str`` — inconsistent categorization among what would otherwise
  all be considered "text" columns. Reason unknown at this point.
- ``Customer ID`` missing count (107,927 rows, ~20.5% of the dataset) — not yet
  assessed for cause or impact.
- ``Description`` missing count (2,928 rows) — not yet assessed for cause or
  impact.

In [ ]:
# Print table with summary statistics of each numerical column: count, mean, 
# std, Range, min, and max.  Each column label is shown. 
transactions_raw.describe()

In [ ]:
# Print table with summary stats for text/categorical columns: 
# count (number of non-null entries), unique (number distinct values), top (mode), and freq (of mode). 
transactions_raw.describe(include= ['object', 'str'])

### Interpretation: ``transactions_raw.describe()`` and ``describe(include=['object','str'])``

#### Numeric columns (Quantity, InvoiceDate, Price, Customer ID)
- ``Quantity``: min -9600, max 19152 — negative values present; mean (10.3) far
  above median (3), suggesting skew/outliers.
- ``Price``: min -53594.36, max 25111.09 — negative values present; mean (4.69)
  far above median (2.10).
- ``InvoiceDate``: range Dec 2009 – Dec 2010; std is NaN (expected — standard
  deviation is not defined for datetime values).
- ``Customer ID``: min 12346, max 18287, count 417,534 (matches non-null count
  from .info()); note this is an identifier stored as a numeric dtype, so
  mean/std are not meaningful despite being computed.

#### Text/categorical columns (Invoice, StockCode, Description, Country)
- ****Counts** match ``.info()`` non-null figures (Description: 522,533; others: 525,461).
- **Unique values**: Invoice 28,816; StockCode 4,632; Description 4,681; Country 40.
- **Mode**: ``Invoice`` 537434 (675 occurrences); ``StockCode`` 85123A (3,516); ``Description``
  "WHITE HANGING HEART T-LIGHT HOLDER" (3,549); ``Country`` United Kingdom (485,852
  — 92.4% of rows).

#### Anomalies flagged for follow-up (not yet investigated)
- ``Quantity`` and ``Price`` both have negative minimums — reason unknown at this point.
- ``Quantity`` and ``Price`` means are well above their medians in both cases —
  possible outliers or skew, not yet assessed.
- ``StockCode`` (4,632) and ``Description`` (4,681) unique counts differ by 49 —
  reason unknown at this point.
- ``Invoice`` 537434 has an unusually high occurrence count (675) relative to
  typical order size — reason unknown at this point.
- ``Country`` distribution is heavily concentrated in one value (92.4% UK) —
  noted for awareness, not yet assessed for impact.

In [ ]:
# See number of unique values accross all columns
transactions_raw.nunique().sort_values()

### Interpretation: ``transactions_raw.nunique().sort_values()``

#### Distinct value counts, per column (ascending)
| Column | Distinct Values |
|---|---|
| Country | 40 |
| Quantity | 825 |
| Price | 1,606 |
| Customer ID | 4,383 |
| StockCode | 4,632 |
| Description | 4,681 |
| InvoiceDate | 25,296 |
| Invoice | 28,816 |

#### Observations
- ``Country``: low cardinality relative to 525,461 rows — clearly categorical.
- ``Customer ID``: 4,383 distinct customers across 417,534 non-null rows —
  averages roughly 95 transactions per customer, though distribution across
  customers is unknown at this point.
- ``StockCode / Description``: cardinality close to each other (4,632 vs. 4,681)
  but not identical.
- ``InvoiceDate``: 25,296 distinct timestamps across 525,461 rows — most
  timestamps are shared by multiple rows, consistent with multiple line
  items per invoice.
- ``Invoice``: highest cardinality of any column (28,816) — consistent with its
  likely role as a near-unique transaction/order identifier.

#### Anomalies flagged for follow-up (not yet investigated)
- StockCode (4,632) and Description (4,681) distinct counts differ by 49 —
  reason unknown at this point.
- InvoiceDate (25,296) is lower than Invoice (28,816) — reason unknown at
  this point (could reflect multiple invoices sharing an identical
  timestamp, or another cause).

## Step 1C:  Eyeball dataset

In [ ]:
transactions_raw.head()

In [ ]:
transactions_raw.tail()

In [ ]:
transactions_raw.sample(10, random_state=2)

## Stage 2 (Audit): Quantify Data Quality Issues

- Determine Null (missing values) rates:
    - Under 5%: safe
    - 5-30%: Moderate risk.  Needs further investigation.
    - Greater than or equal to 30%: High risk.  Column may be unusable. 
- Determine duplicate row rates
- Perform value range assessment if needed
- Check for overlap/consistency between proposed conditions

In [ ]:
# Stage 2 (Audit): initialize the config dictionary.
# Populated incrementally, one entry at a time, as each audit finding below
# is confirmed worth carrying forward into Stage 3/4 — never reconstructed later.
EXPECTED = {}


In [ ]:
# Check data for null values. This is done column by column.
# Note: Aggregating a DataFrame along its default axis (columns)
# always produces a Series whose index is the original column names.
null_counts = transactions_raw.isnull().sum()
print(null_counts)
print()

# Determine null rates
null_pct_desc = null_counts['Description'] / len(transactions_raw) * 100
null_pct_id = null_counts['Customer ID'] / len(transactions_raw) * 100
# Print results
print(f"Description null rate: {null_pct_desc:.1f}%")
print()
print(f"Customer ID null rate: {null_pct_id:.1f}%")

### Interpretation: transactions_raw.isnull().sum() and null rate calculation

#### Null counts, by column
`transactions_raw.isnull().sum()`

| Column | Null Count |
|---|---|
| Invoice | 0 |
| StockCode | 0 |
| Description | 2,928 |
| Quantity | 0 |
| InvoiceDate | 0 |
| Price | 0 |
| Customer ID | 107,927 |
| Country | 0 |

#### Null rates
`.isna().mean() * 100`, for the two columns with missing values

- Description: 0.6%
- Customer ID: 20.5%

In [ ]:
# Add Description and Customer ID null counts to config dictionary. 
EXPECTED['description_nulls'] = null_counts['Description']
EXPECTED['customer_id_nulls'] = null_counts['Customer ID']

In [ ]:
# Determine duplicate row rates
dup_count = transactions_raw.duplicated().sum()
print(f"Duplicate rows: {dup_count} ({dup_count/len(transactions_raw)*100:.1f}%)")


### Interpretation: transactions_raw.duplicated().sum()
#### Result
6,865 duplicate rows (1.3% of 525,461 total rows)
#### Anomalies flagged for follow-up (not yet investigated)
- Reason for the duplication is unknown at this point.

In [ ]:
# Add duplicate rows count to config dictionary
EXPECTED['duplicate_rows'] = dup_count

### Perform value range assessment: Audit Negative Values in ``Quantity`` and ``Price``

#### The Prompt
transactions_raw.describe() showed Quantity min = -9600, Price min = -53594.36
— flagged as unexplained anomalies, not yet investigated.

#### Purpose
Determine what negative values in each column represent, and how frequently
they occur, before characterizing them as valid or invalid.

In [ ]:
# Determine negative "Quantity" rate 
neg_qty_count = (transactions_raw['Quantity'] < 0).sum()
neg_qty_pct = (transactions_raw['Quantity'] < 0).mean() * 100

# Determine negative "Price" rate
neg_price_count = (transactions_raw['Price'] < 0).sum()
neg_price_pct = (transactions_raw['Price'] < 0).mean() * 100

print(f"Negative Quantity: {neg_qty_count} rows ({neg_qty_pct:.1f}%)")
print(f"Negative Price: {neg_price_count} rows ({neg_price_pct:.1f}%)")


In [ ]:
# Check upper-tail distribution (90th/95th/99th percentile) for Quantity and Price
print(transactions_raw['Quantity'].describe(percentiles=[.9, .95, .99]))
print()
print(transactions_raw['Price'].describe(percentiles=[.9, .95, .99]))

### Interpretation: Negative Value Rates and Upper-Tail Distribution (``Quantity``, ``Price``)

#### Negative value rates
- ``Quantity``: 12,326 rows negative (2.3%)
- ``Price``: 3 rows negative (0.0%)

#### Upper-tail distribution — ``Quantity``
| Percentile | Value |
|---|---|
| 90% | 24 |
| 95% | 30 |
| 99% | 120 |
| max | 19,152 |

#### Upper-tail distribution — ``Price``
| Percentile | Value |
|---|---|
| 90% | 7.95 |
| 95% | 10.17 |
| 99% | 19.95 |
| max | 25,111.09 |

#### Observations
- Negative ``Quantity`` (2.3%) occurs at a rate roughly 77x higher than
  negative ``Price`` (0.0%, 3 rows total).
- Both columns show a large jump between the 99th percentile and the max:
  Quantity goes from 120 (99th) to 19,152 (max); Price goes from 19.95
  (99th) to 25,111.09 (max).

#### Anomalies flagged for follow-up (not yet investigated)
- Reason for negative ``Quantity`` values (12,326 rows) — not yet investigated.
- Reason for negative ``Price`` values (3 rows) — not yet investigated.
- Cause of the single maximum value far above the 99th percentile, in both
  ``Quantity`` and ``Price`` — not yet investigated; unknown whether this reflects
  one extreme row, a handful of large orders, or another pattern.

In [ ]:
# Add negative values counts for Quantity and Price to config dictionary
EXPECTED['negative_quantity_rows'] = neg_qty_count
EXPECTED['negative_price_rows'] = neg_price_count

### Perform Cross Column Consistency Check
#### Consistency Check: ``StockCode`` vs. ``Description``

#### Prompt: 
``.nunique()`` showed ``StockCode`` = 4,632 distinct values, ``Description`` = 4,681 distinct values — a gap of 49, suggesting these two columns do not track together 1:1 as expected.

In [ ]:
# Step 1: confirm the two counts directly, from a single source
stockcode_nunique = transactions_raw['StockCode'].nunique()
description_nunique = transactions_raw['Description'].nunique()

print(f"Distinct StockCode values: {stockcode_nunique}")
print(f"Distinct Description values: {description_nunique}")
print(f"Difference: {description_nunique - stockcode_nunique}")

In [ ]:
# Step 2: how many Description values exist per StockCode
descriptions_per_stockcode = transactions_raw.groupby('StockCode')['Description'].nunique()

# Step 3: isolate StockCodes with more than one associated Description
affected = descriptions_per_stockcode[descriptions_per_stockcode > 1]

print(f"\nStockCodes with more than one Description: {len(affected)}")
print(f"Out of {len(descriptions_per_stockcode)} total StockCodes")

# Step 4: view the most affected StockCodes, worst first
affected.sort_values(ascending=False)


### Audit Finding: Cross-Column Consistency Check — StockCode vs. Description

#### Prompt
`.nunique()` showed StockCode = 4,632 distinct values, Description = 4,681
distinct values — a difference of 49.

#### Results
- StockCodes with more than one associated Description: **687**
- Out of 4,632 total StockCodes (**14.8%**)

#### Top affected StockCodes, by Description variant count
| StockCode | Distinct Descriptions |
|---|---|
| 22423 | 6 |
| 22734 | 5 |
| 22139 | 4 |
| 21523 | 4 |
| 20685 | 4 |
| ... | ... |
| 85232b | 2 |
| 21007 | 2 |
| BANK CHARGES | 2 |
| DCGSSBOY | 2 |
| 16012 | 2 |

#### Observation
The raw cardinality gap (49) significantly understates the scope of the
inconsistency — 687 StockCodes are actually affected, roughly 14x the
number the initial gap suggested. This is possible because a StockCode
with, say, 3 Descriptions and another StockCode that's missing entirely
from Description's count can partially cancel out in a simple difference
of two totals; the difference-of-counts figure does not by itself reveal
how many individual keys are involved.

#### Anomalies flagged for follow-up (not yet investigated)
- Root cause confirmed for only one StockCode so far (22423 — condition/
  damage notes mixed into Description, from a prior drill-down). Whether
  this same cause explains the remaining 686 affected StockCodes is not
  yet established.
- Two non-numeric-looking StockCode values appear in this list
  ("BANK CHARGES", "DCGSSBOY", "85232b") — inconsistent with the
  alphanumeric product-code pattern seen elsewhere (e.g. 85123A). Reason
  and prevalence not yet investigated.

In [ ]:
# Add number Stockcodes with more than one description to config dictionary
# Add total number of stockcodes to config dictionary
EXPECTED['stockcodes_with_multiple_descriptions'] = len(affected)
EXPECTED['total_stockcodes'] = len(descriptions_per_stockcode)

###Stage 2C (Audit): Confirm the Config Dictionary

`EXPECTED` was built incrementally above, one entry at a time, at each audit
finding — nothing is reconstructed here. This step only confirms the finished
dictionary before Stage 3 begins.


In [ ]:
# Stage 2C: confirm EXPECTED is complete before moving into Stage 3
# Log and print config dictionary
logger.info("Stage 2 audit complete. EXPECTED config:")
for key, value in EXPECTED.items():
    logger.info(f"  {key}: {value}")


## Stage 3 : Cleaning Transformations plan per finding

#### 1. ``Description`` — missing values
- **Finding**: 2,928 rows (0.6%) with Description = NaN.
- **Strategy**: Drop rows.

#### 2. ``Customer ID`` — missing values
- **Finding**: 107,927 rows (20.5%) with Customer ID = NaN.
- **Strategy**: Keep all rows in the master table. Exclude only at the point of customer-level aggregation; do not drop or impute.

#### 3. Duplicate rows
- **Finding**: 6,865 full-row duplicates (1.3%).
- **Strategy**: Keep. No basis to distinguish legitimate repeat line items from a loading artifact without external context; dropping unverified risks discarding real transactions.

#### 4. ``Quantity`` — negative values
- **Finding**: 12,326 rows (2.3%) with Quantity < 0.
- **Strategy**: Separate into a ``returns`` table. Established as the dataset's signal for returns, not invalid data.

#### 5. ``Price`` — negative values
- **Finding**: 3 rows with Price < 0.
- **Strategy**: Remove outright. Negligible count; no legitimate meaning established for a negative price.

#### 6. ``Quantity`` and ``Price`` — upper-tail maximum values
- **Finding**: Both columns show a large jump between the 99th percentile and the max (Quantity: 120 → 19,152; Price: 19.95 → 25,111.09).
- **Strategy**: Keep, no transformation applied. No basis to distinguish a legitimate bulk order from a data artifact without external context; capping or removing would be an unsupported assumption.

#### 7. ``StockCode`` — mixed dtype
- **Finding**: Column mixes ``int`` and ``str`` values under object dtype (e.g. 22423 as int, '85123A' as str).
- **Strategy**: Convert entire column to string type, stripped of whitespace.

#### 8. ``Invoice``, ``StockCode``, ``Description`` — object vs. str dtype label
- **Finding**: These three columns show dtype ``object`` while Country shows ``str``.
- **Strategy**: No cleaning action required; attributable to pandas 3.0's string-dtype migration (Pandas4Warning), not a data quality issue. Apply the same _string normalization_ as ``StockCode`` (see #7) for consistency.

#### 9. ``StockCode`` / ``Description`` — cardinality mismatch
- **Finding**: 687 StockCodes (14.8%) have more than one associated Description value.
- **Strategy**: Derive one canonical Description per StockCode, using the most frequently occurring value for that code; overwrite the column with this mapped value.

#### 10. ``StockCode`` — non-product values
- **Finding**: Values such as "BANK CHARGES" and "DCGSSBOY" appear in StockCode, inconsistent with the alphanumeric product-code pattern seen elsewhere.
- **Strategy**: Keep, no transformation applied. No basis to confirm these are erroneous versus legitimate non-product line items without external context.

#### 11. Invoice 537434 — outlier line-item count
- **Finding**: 675 line items on a single invoice, the highest in the dataset.
- **Strategy**: Keep, no transformation applied. A single large invoice is not evidence of an error on its own.

#### 12. ``Description`` — blank/whitespace values distinct from NaN
- **Finding**: Not yet quantified; .isnull() only captures true NaN, and an empty or whitespace-only string would not be counted by it.
- **Strategy**: Quantify directly (count of empty/whitespace Description values) as a first coding step; apply the same drop strategy as #1 if any are found.

## Stage 4: Apply Cleaning Transformations and Verify Result

Each finding below is applied as its own cell, following one consistent pattern:
compute before/after counts, apply the transformation, verify with `assert`
against the `EXPECTED` config dictionary (never a hardcoded number), and log
the outcome with `logger.info(...)`.


In [ ]:
# Stage 4, Finding #1: Description — missing values
# Finding: EXPECTED['description_nulls'] rows with Description = NaN (see Stage 2 audit).
# Strategy: Drop rows.
# Function defined

def drop_null_rows(df, column, expected_count):
    # This comment is diplayed when help(drop_null_rows) is called
    """Drops rows where `column` is null; verifies the drop count matches expected_count."""
    before = len(df)
    df = df.dropna(subset=[column])
    after = len(df)

    assert before - after == expected_count, \
        f"Expected {expected_count} rows dropped from {column}, got {before - after}"
    assert df[column].isna().sum() == 0, \
        f"{column} column still contains nulls after dropna"

    logger.info(f"Dropped {before - after} rows with missing {column}. Column is now null-free.")
    return df

In [ ]:
# Stage 4, Finding #1: Description — missing values
# Drop rows of missing values
transactions_raw = drop_null_rows(transactions_raw, 'Description', EXPECTED['description_nulls'])